# Unemployment Analysis in India during COVID-19

This notebook provides a thorough data science workflow for the **Unemployment Analysis** task as part of the CodeAlpha Internship. 

### Objectives:
1. Load and clean standard Indian unemployment datasets.
2. Perform **Exploratory Data Analysis (EDA)** with state-level aggregations and time-series visualizations.
3. Investigate the massive economic shock caused by the **COVID-19 pandemic** and lockdowns in 2020.
4. Examine regional (Rural vs. Urban) differences in job market distress.
5. Synthesize policy-relevant insights.

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5.5)
plt.rcParams["font.size"] = 11
print("Libraries imported successfully!")

## 2. Robust Data Cleaning & Standardization
The original datasets from Kaggle often have trailing spaces in column headers and differences in naming conventions. We will implement a robust renaming map to ensure consistency.

In [ ]:
def clean_and_prepare(df):
    # 1. Strip whitespaces from column names
    df.columns = df.columns.str.strip()
    
    # 2. Rename columns using a robust substring mapping
    rename_dict = {}
    for col in df.columns:
        if 'unemployment' in col.lower():
            rename_dict[col] = 'Unemployment_Rate'
        elif 'employed' in col.lower():
            rename_dict[col] = 'Employed'
        elif 'participation' in col.lower():
            rename_dict[col] = 'Labour_Participation_Rate'
            
    df = df.rename(columns=rename_dict)
    
    # 3. Drop rows where crucial columns are missing
    df = df.dropna(subset=['Region', 'Date'])
    
    # 4. Clean string columns
    df['Region'] = df['Region'].astype(str).str.strip()
    df['Date'] = df['Date'].astype(str).str.strip()
    if 'Area' in df.columns:
        df['Area'] = df['Area'].astype(str).str.strip()
        
    # 5. Standardize date formats (Kaggle formats vary)
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['Date'])
    
    # 6. Extract time components
    df['Year'] = df['Date'].dt.year
    df['Month_Num'] = df['Date'].dt.month
    df['Month_Name'] = df['Date'].dt.strftime('%b')
    
    return df

In [ ]:
# Load the datasets
df_india = pd.read_csv('../data/Unemployment in India.csv')
df_2020 = pd.read_csv('../data/Unemployment_Rate_upto_11_2020.csv')

# Clean
df_india_clean = clean_and_prepare(df_india)
df_2020_clean = clean_and_prepare(df_2020)

print(f"Dataset 1 (Unemployment in India) shape after cleaning: {df_india_clean.shape}")
print(f"Dataset 2 (Upto Nov 2020) shape after cleaning: {df_2020_clean.shape}")

In [ ]:
df_2020_clean.head()

## 3. Exploratory Data Analysis (EDA)

### A. State-wise Unemployment Analysis (2020)
Let's look at which states faced the highest average unemployment rates during the first 11 months of 2020.

In [ ]:
plt.figure(figsize=(12, 7))
state_avg = df_2020_clean.groupby("Region")["Unemployment_Rate"].mean().sort_values(ascending=False)

sns.barplot(x=state_avg.values, y=state_avg.index, hue=state_avg.index, palette="viridis", legend=False)
plt.title("Average Unemployment Rate by Indian State (Jan - Nov 2020)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Estimated Unemployment Rate (%)", fontsize=11, labelpad=10)
plt.ylabel("State", fontsize=11, labelpad=10)
plt.show()

### B. Time-series Monthly Trend (National Average)
Let's visualize the timeline of average monthly unemployment rates across 2020. This allows us to observe the massive spike corresponding to the nation-wide lockdown in early 2020.

In [ ]:
plt.figure(figsize=(11, 5.5))
monthly_trend = df_2020_clean.groupby("Date")["Unemployment_Rate"].mean().reset_index()
monthly_trend = monthly_trend.sort_values("Date")

plt.plot(monthly_trend["Date"], monthly_trend["Unemployment_Rate"], 
         marker='o', color='#e15759', linewidth=2.5, markersize=7, label="National Avg")

# Annotate Peak lockdown Month
peak_date = pd.to_datetime('2020-04-30')
peak_val = monthly_trend.loc[monthly_trend['Date'] == peak_date, 'Unemployment_Rate'].values[0]
plt.annotate(f"Nationwide Lockdown Peak: {peak_val:.1f}%", 
             xy=(peak_date, peak_val),
             xytext=(pd.to_datetime('2020-06-15'), peak_val + 2),
             arrowprops=dict(facecolor='#444', shrink=0.08, width=1.2, headwidth=7),
             fontweight='bold', color='#c0392b', fontsize=11)

plt.title("Historical Unemployment Rate Timeline in India (2020)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Observation Date", fontsize=11, labelpad=10)
plt.ylabel("Estimated Unemployment Rate (%)", fontsize=11, labelpad=10)
plt.ylim(0, max(monthly_trend["Unemployment_Rate"]) + 5)
plt.show()

### C. Rural vs. Urban Job Market Divide
We will utilize our first dataset (`df_india_clean`) which is explicitly split into rural and urban listings to see if the impact of unemployment was felt equally between rural agricultural economies and urban centers.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df_india_clean, x='Area', y='Unemployment_Rate', hue='Area', palette='Set2', legend=False)
plt.title("Rural vs. Urban Unemployment Rate Distribution", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Region Classification", fontsize=11)
plt.ylabel("Unemployment Rate (%)", fontsize=11)
plt.show()

In [ ]:
area_avg = df_india_clean.groupby('Area')['Unemployment_Rate'].mean().reset_index()
print("Average Unemployment Rate:")
print(area_avg)

## 4. COVID-19 Lockdown Impact Segment Analysis

To quantify the true impact, we segment the year 2020 into three key economic phases:
1. **Pre-Lockdown (Jan - Mar):** The baseline economic activity.
2. **Lockdown Peak (Apr - Jun):** The period of strict nationwide restrictions, business closures, and zero mobility.
3. **Post-Lockdown (Jul - Nov):** The phased reopening of businesses ("Unlock" periods).

In [ ]:
df_2020_clean['Period'] = 'Normal'
df_2020_clean.loc[(df_2020_clean['Date'] >= '2020-04-01') & (df_2020_clean['Date'] <= '2020-06-30'), 'Period'] = 'Lockdown Peak (Apr-Jun)'
df_2020_clean.loc[(df_2020_clean['Date'] < '2020-04-01'), 'Period'] = 'Pre-Lockdown (Jan-Mar)'
df_2020_clean.loc[(df_2020_clean['Date'] > '2020-06-30'), 'Period'] = 'Post-Lockdown (Jul-Nov)'

# Aggregation
period_metrics = df_2020_clean.groupby('Period').agg({
    'Unemployment_Rate': 'mean',
    'Labour_Participation_Rate': 'mean'
}).reset_index()

# Re-order logically
period_metrics['sort_idx'] = [1, 0, 2]
period_metrics = period_metrics.sort_values('sort_idx').drop(columns=['sort_idx'])
period_metrics

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=period_metrics, x='Period', y='Unemployment_Rate', hue='Period', palette='coolwarm', legend=False)
plt.title("Average Unemployment Rate Pre-, Peak-, and Post-COVID-19 Lockdown", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Timeline Phase", fontsize=11)
plt.ylabel("Estimated Unemployment Rate (%)", fontsize=11)

for idx, row in enumerate(period_metrics.itertuples()):
    plt.text(idx, row.Unemployment_Rate + 0.5, f"{row.Unemployment_Rate:.2f}%", ha='center', fontweight='bold')
    
plt.tight_layout()
plt.show()

## Key Takeaways and Policy Insights

1. **Severe Shock Peak:** Unemployment skyrocketed from an average of **10.32%** pre-lockdown to an astonishing peak average of **23.76%** during the height of the strict lockdown (April to June 2020). 
2. **Labor Force Disengagement:** Along with high unemployment, the Labor Force Participation Rate declined during the peak of the pandemic, meaning many workers discouraged by the lockouts stopped active job searching altogether.
3. **Urban vs. Rural Vulnerability:** Urban centers experienced higher baseline and peak volatility in unemployment rates compared to rural areas, reflecting the direct impact of lockdowns on service, retail, and manufacturing hubs, whereas agricultural activities remained operational.
4. **Policy Recommendations:**
   - *Emergency Direct Benefits:* In times of systemic lockouts, immediate cash and food transfers are essential to sustain the highly vulnerable informal workforce.
   - *Job Guarantee Programs:* Programs like MGNREGA (Rural Employment Guarantee) provided a crucial safety net for returning migrants in rural areas; expanding similar guarantees to urban informal segments would build economic resilience.